In [ ]:
''' 
Variance assessment to estimate k
'''

import json 
import matplotlib.pyplot as plt 
import seaborn as sns
import numpy as np 
import random 
import importlib 
import pandas as pd 
import os 
import analysis_utils
import constants
from scipy import stats
import matplotlib.gridspec as gridspec
from scipy.stats import wasserstein_distance, entropy
from sklearn.linear_model import LinearRegression
from matplotlib.ticker import ScalarFormatter
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr


base_game_objs, idx2game, game2idx, game_stimuli = analysis_utils.process_game_stimuli(constants.THINK_STIMULI_PTH)

human_df = pd.read_csv(constants.THINK_HUMAN_DATA)

main_dir = constants.PROCESSED_RES_DIR
fig_data_file_pth = constants.FIG_DATA_DIR

import matplotlib as mpl 
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
'''
Think data
''' 
with open(f"{main_dir}think/human_processed.json") as f:
    h = json.load(f)
with open(f"{main_dir}think/model_payoff.json") as f:
    model_payoff = json.load(f)
think_data = {
    "human_payoff": h["human_payoff"],
    "human_fun":    h["human_fun"],
    "model_payoff": model_payoff,
}


In [ ]:
ordered_games = sorted(think_data['human_payoff'])

In [ ]:

def get_human_model_variances(k, task, model_name,
                              n_sim_participants=20):
    """Return Var_human and Var_model-sample for **each stimulus**."""
    human_var   = []
    model_var   = []
    stimulus_ids = []

    for game_id in ordered_games:
        game_df = human_df.query("game_id == @game_id and task == 'advantage'")
        if game_df.empty:
            continue

        raw = []
        human_scores =  eval(game_df.iloc[0].all_scores)
        for score_entry in human_scores: 
            draw = score_entry["draw_response"]   / 100.0
            p1   = score_entry["firstplayer_response"] / 100.0
            pwin = p1 * (1 - draw)
            util = analysis_utils.compute_utility(draw, pwin)

            if   task == "advantage-draw":     raw.append(draw)
            elif task == "advantage-win":      raw.append(pwin)
            elif task == "advantage-utility":  raw.append(util)

        if len(raw) < 2:        # need at least two responses for a variance
            continue

        scores = raw
        human_var.append(np.var(scores))

        # --- model side ----------------------------------------------------
        sims = sum(think_data['model_payoff'][model_name][game_id]['sims'], [])
        # bootstrap k * n participants with replacement
        boot = np.random.choice(sims, k * n_sim_participants, replace=True)
        groups = analysis_utils.by_participants(list(boot),
                                                n=n_sim_participants, m=k)

        sim_scores = []
        for g in groups:
            # same mapping as above
            pwin_sim = g.count(1) / len(g)
            pdraw_sim = g.count(0) / len(g)
            util_sim = analysis_utils.map_score_to_judgment([g])[0]['exp_util']

            if   task == "advantage-draw":     sim_scores.append(pdraw_sim)
            elif task == "advantage-win":      sim_scores.append(pwin_sim)
            elif task == "advantage-utility":  sim_scores.append(util_sim)

        model_var.append(np.var(sim_scores))
        stimulus_ids.append(game_id)

    return np.array(human_var), np.array(model_var), stimulus_ids


def estimate_k_and_omega(v_human, v_model):
    reg = LinearRegression().fit(v_model.reshape(-1, 1), v_human)
    a, b = reg.coef_[0], reg.intercept_
    k_hat = 1/a if a != 0 else np.nan 
    omega = np.sqrt(b) if b > 0 else np.nan
    return k_hat, omega, a, b, reg.score(v_model.reshape(-1, 1), v_human)


# loop over varied k (num sims)
k_grid = [2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 50]


n_bootstrap = 1000

np.random.seed(7)




In [ ]:
for model_name in ['ours_full', 'ours']: 

    records = {k: [] for k in k_grid}

    for k in k_grid:
        # repeat bootstrap
        boot_records = []

        for _ in range(n_bootstrap): 
            # get bootstrapped variances for that k 
            v_h, v_m, stim_ids = get_human_model_variances(k,
                                                        task="advantage-utility",
                                                        model_name=model_name,
                                                        n_sim_participants=20)

            boot_records.append(dict(k_boot=k,

                                h=v_h,
                                m=v_m))

        records[k] = boot_records
    k_grid_sub = k_grid

    n_quintiles = 10
    shared_max_x = 0.35

    fig = plt.figure(figsize=(30, 5))
    outer = gridspec.GridSpec(1, len(k_grid_sub), wspace=0.3, hspace=0.2)

    mses_per_k = []
    mses_ci_low = []
    mses_ci_high = []

    mses_per_k = []



    save_viz_data = {'k': [], 
                    'main_scatter': [], 'bars': []}

    # plotting of multi-metric
    for i, k in enumerate(k_grid_sub):
        gs = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=outer[i],
                                            height_ratios=[1, 1], hspace=0.05)
        ax_main = plt.Subplot(fig, gs[0])
        ax_hist = plt.Subplot(fig, gs[1],)# sharex=ax_main)
        
        k_rec = records[k]
        all_h = np.array([entry['h'] for entry in k_rec])
        all_m = np.array([entry['m'] for entry in k_rec])
        h_mean = np.mean(all_h, axis=0)
        m_mean = np.mean(all_m, axis=0)

        eps = 1e-8                     # avoids divide-by-zero
        rel_err  = (all_h - all_m) / (all_h + eps)     # shape (B, games)
        bootstrap_rrmse = np.sqrt(np.mean(rel_err**2, axis=1))
        
        eps = 1e-6
        log_err = np.log(all_h + eps) - np.log(all_m + eps)
        bootstrap_rmsle = np.sqrt(np.mean(log_err**2, axis=1))
        
        bootstrap_rmse = np.sqrt(np.mean((all_h - all_m)**2, axis=1))
        range_h = np.ptp(all_h)
        std_h = np.std(all_h)
        mean_h = np.mean(all_h)

        nrmse_range = bootstrap_rmse / range_h
        nrmse_std = bootstrap_rmse / std_h
        nrmse_mean = bootstrap_rmse / mean_h  # if you want percentage of mean



        mean_mse = np.mean(nrmse_mean)
        ci_low = np.percentile(nrmse_mean, 2.5)
        ci_high = np.percentile(nrmse_mean, 97.5)

        mses_per_k.append(mean_mse)
        mses_ci_low.append(mean_mse - ci_low)
        mses_ci_high.append(ci_high - mean_mse)


        # Raw scatter
        ax_main.scatter(m_mean, h_mean, s=14, alpha=0.3, color='tab:blue', label='Raw Points')
        line_max = max(0.5, np.max(m_mean), np.max(h_mean))
        ax_main.plot([0, line_max], [0, line_max], '--', color='red', linewidth=3, zorder=1)

        save_viz_data['main_scatter'].append([[m_mean, h_mean]])

        # Equal-sized group binning
        sorted_idx = np.argsort(m_mean)
        group_size = len(m_mean) // n_quintiles
        inds = np.empty_like(m_mean, dtype=int)
        for b in range(n_quintiles):
            start = b * group_size
            end = (b + 1) * group_size if b < n_quintiles - 1 else len(m_mean)
            inds[sorted_idx[start:end]] = b

        means, cis_low, cis_high, counts, bin_mses, bin_centers = [], [], [], [], [], []
        for b in range(n_quintiles):
            in_bin = (inds == b)
            vals_h = h_mean[in_bin]
            vals_m = m_mean[in_bin]
            counts.append(np.sum(in_bin))
            if len(vals_h) > 0:
                mean_h = np.mean(vals_h)
                std_h = np.std(vals_h)
                mean_m = np.mean(vals_m)
                means.append(mean_h)
                cis_low.append(std_h)
                cis_high.append(std_h)
                bin_centers.append(mean_m)
                bin_mses.append((mean_h - mean_m) ** 2)
            else:
                means.append(np.nan)
                cis_low.append(np.nan)
                cis_high.append(np.nan)
                bin_centers.append(np.nan)


        means = np.array(means)
        cis_low = np.array(cis_low)

        ax_main.errorbar(bin_centers, means, yerr=cis_low, fmt='o',
                        color='black', capsize=2, ms=7, label='Mean ± 95% CI', alpha=0.5)
        
        save_viz_data['bars'].append([[bin_centers, means, cis_low]])
        save_viz_data['k'].append(k)
        
        #ax_main.set_xlim(0, shared_max_x)
        ax_main.set_title(f"k = {k}", fontsize=14)
        if i == 0:
            ax_main.set_ylabel("Human Variance", fontsize=13)

        ax_main.set_xlabel("Model Variance", fontsize=13)
        ax_main.set_xlim([0, np.max(m_mean) + 0.05])
        #plt.setp(ax_main.get_xticklabels(), visible=False)

        fig.add_subplot(ax_main)
        # fig.add_subplot(ax_hist)

    plt.tight_layout()
    plt.savefig(f"final_figures/variance_model_{model_name}_k.svg", dpi=400)
    plt.show()

    save_viz_data = pd.DataFrame(save_viz_data)
    save_viz_data.to_csv(fig_data_file_pth + f'{model_name}_vary_k_individ_stats.csv')


    

    # store both point‐estimates and CI‐widths
    results = []

    for k in k_grid_sub:
        recs = records[k]
        all_h = np.stack([r['h'] for r in recs], axis=0)  # shape (B, G)
        all_m = np.stack([r['m'] for r in recs], axis=0)

        # point‐estimate per game
        h_hat = all_h.mean(axis=0)
        m_hat = all_m.mean(axis=0)


        bs_rmse = np.sqrt(np.mean((all_h - all_m)**2, axis=1))
        print(np.shape(bs_rmse))
        ci_lo_rmse, ci_hi_rmse = np.percentile(bs_rmse, [2.5, 97.5])

        pearson_r, _   = stats.pearsonr(m_hat, h_hat)
        spearman_rho,_ = stats.spearmanr(m_hat, h_hat)
        lr = LinearRegression().fit(m_hat.reshape(-1,1), h_hat)
        alpha, beta = lr.intercept_, lr.coef_[0]
        miscal = abs(alpha) + abs(beta - 1)
        diffs = h_hat - m_hat
        rmse  = np.mean(bs_rmse) 
        mae   = np.mean(np.abs(diffs))
        mape  = np.mean(np.abs(diffs) / (h_hat + 1e-8))

        minv, maxv = h_hat.min(), h_hat.max()
        bins = np.linspace(minv, maxv, 10)
        p_hist,_ = np.histogram(h_hat, bins=bins, density=True)
        q_hist,_ = np.histogram(m_hat, bins=bins, density=True)
        p_hist += 1e-8; q_hist += 1e-8
        kl = entropy(p_hist, q_hist)

        # compute WD over hists
        hist1, edges = np.histogram(h_hat, bins=10, range=(minv,maxv), density=True)
        hist2,_      = np.histogram(m_hat, bins=10, range=(minv,maxv), density=True)
        centers = (edges[:-1]+edges[1:])/2
        
        # Wasserstein per bootstrap
        # note: this loops over B replicates
        bs_wdist = []
        for h_hat_b, m_hat_b in zip(all_h, all_m): 
            hist1, edges = np.histogram(h_hat_b, bins=10, range=(minv,maxv), density=True)
            hist2,_      = np.histogram(m_hat_b, bins=10, range=(minv,maxv), density=True)
            centers = (edges[:-1]+edges[1:])/2
            wdist = wasserstein_distance(centers, centers, hist1, hist2)
            bs_wdist.append(wdist)
        bs_wdist = np.array(bs_wdist)
        ci_lo_w, ci_hi_w = np.percentile(bs_wdist, [2.5, 97.5])
        
        
        wdist = np.mean(bs_wdist)

        results.append({
            'k': k,
            'pearson_r': pearson_r,
            'spearman_rho': spearman_rho,
            'alpha': alpha,
            'beta': beta,
            'miscalibration': miscal,
            'rmse': rmse,
            'rmse_lo': ci_lo_rmse,
            'rmse_hi': ci_hi_rmse,
            'wasserstein': wdist,
            'wdist_lo': ci_lo_w,
            'wdist_hi': ci_hi_w,
            'mae': mae,
            'mape': mape,
            'kl_divergence': kl
        })

    df = pd.DataFrame(results).set_index('k').sort_index()

    ks      = df.index.values
    rmse    = df['rmse'].values

    ci_lo   = df['rmse_lo'].values
    ci_hi   = df['rmse_hi'].values


    err_lo  = np.abs(rmse - ci_lo)
    err_hi  = np.abs(ci_hi - rmse)

    wdist   = df['wasserstein'].values
    w_lo    = np.abs(wdist - df['wdist_lo'].values)
    w_hi    = np.abs(df['wdist_hi'].values - wdist)



    fig, ax1 = plt.subplots(figsize=(5,4))

    ax2 = ax1.twinx()

    # plot rmse
    ax1.errorbar(ks, rmse, yerr=[err_lo, err_hi],
                fmt='-o', capsize=4, label='RMSE', alpha=0.7, linewidth=2,
                color='black')

    # also plot WD
    ax2.errorbar(ks, wdist, yerr=[w_lo, w_hi],
                fmt='-s', capsize=4, label='Wasserstein Distance',
                alpha=0.7, linewidth=2, 
                color='grey')

    save_viz_data = pd.DataFrame({'k': ks, 
                    'rmse': rmse, 'rmse_err_low': err_lo, 'rmse_err_high': err_hi,
                    'wdist': wdist, 'wdist_err_low': w_lo, 'wdist_err_high': w_hi})

    save_viz_data.to_csv(fig_data_file_pth + f'{model_name}_vary_k_stats.csv')

    ax1.set_xscale('log')
    ax1.set_xlim(1.8, 60)
    tick_vals = [2, 4, 6, 8, 10, 20, 50]
    ax1.set_xticks(tick_vals)
    ax1.xaxis.set_major_formatter(ScalarFormatter())
    ax1.ticklabel_format(axis='x', style='plain')

    ax1.set_xlabel('Number of Simulations (log scale)', fontsize=20)
    ax1.set_ylabel('RMSE', fontsize=20)
    ax2.set_ylabel('Wasserstein Distance', fontsize=20)

    # combined legend
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1+h2, l1+l2, loc='upper right', fontsize=12)
    plt.savefig(f"final_figures/variance_{model_name}_measures.svg", dpi=400)
    plt.tight_layout()
    plt.show()







